# EDA - Extended Yale B Harsh vs Enhanced

Notebook nay chi doc hai bundle Yale B da duoc split san de so sanh ban dieu kien khac nghiet va ban da duoc cai thien preprocessing.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

if not (ROOT / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.process import load_processed_dataset_bundle_for_preset
from src.utils import plot_sample_images


#### Đọc toàn bộ dataset để tính toán thông số độ sáng của từng ảnh

In [ ]:
config = {
    "dataset_name": "extended_yale_b",
    "presets": {
        "harsh": "harsh_conditions",
        "enhanced": "enhanced_conditions",
    },
    "n_samples": 10,
}

pd.Series({
    "dataset_name": config["dataset_name"],
    "harsh_preset": config["presets"]["harsh"],
    "enhanced_preset": config["presets"]["enhanced"],
})


## Load Bundles


In [ ]:
harsh_bundle = load_processed_dataset_bundle_for_preset(
    dataset_name=config["dataset_name"],
    preset_name=config["presets"]["harsh"],
)
enhanced_bundle = load_processed_dataset_bundle_for_preset(
    dataset_name=config["dataset_name"],
    preset_name=config["presets"]["enhanced"],
)

image_shape = harsh_bundle["image_shape"]
image_shape


In [ ]:
comparison_df = pd.DataFrame([
    {
        "variant": "harsh",
        "samples_total": harsh_bundle["summary"]["samples_total"],
        "classes_total": harsh_bundle["summary"]["classes_total"],
        "train_shape": harsh_bundle["summary"]["train_shape"],
        "test_shape": harsh_bundle["summary"]["test_shape"],
        "processing_profile": harsh_bundle["summary"].get("processing_profile"),
        "quality_gate": harsh_bundle["summary"].get("quality_gate"),
        "rejected_extreme_samples": harsh_bundle["summary"]["processing_stats"].get("rejected_extreme_samples", 0),
        "bundle_dir": harsh_bundle["output_dir"],
    },
    {
        "variant": "enhanced",
        "samples_total": enhanced_bundle["summary"]["samples_total"],
        "classes_total": enhanced_bundle["summary"]["classes_total"],
        "train_shape": enhanced_bundle["summary"]["train_shape"],
        "test_shape": enhanced_bundle["summary"]["test_shape"],
        "processing_profile": enhanced_bundle["summary"].get("processing_profile"),
        "quality_gate": enhanced_bundle["summary"].get("quality_gate"),
        "rejected_extreme_samples": enhanced_bundle["summary"]["processing_stats"].get("rejected_extreme_samples", 0),
        "bundle_dir": enhanced_bundle["output_dir"],
    },
]).set_index("variant")

comparison_df


## Subject Distribution


In [ ]:
harsh_counts = pd.Series(harsh_bundle["metadata"]["subject_names"], name="harsh").value_counts().sort_index()
enhanced_counts = pd.Series(enhanced_bundle["metadata"]["subject_names"], name="enhanced").value_counts().sort_index()
subject_counts_df = pd.concat([harsh_counts, enhanced_counts], axis=1)
subject_counts_df.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
subject_counts_df.plot(kind="bar", ax=ax, width=0.85)
ax.set_title("Images per Subject in Processed Bundles")
ax.set_xlabel("Subject")
ax.set_ylabel("Images")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()


## Pixel Statistics After Processing


#### Đi sâu vào phân tích biến đổi ánh sáng 1 người 

In [ ]:
harsh_mean = harsh_bundle["X"].mean(axis=1)
enhanced_mean = enhanced_bundle["X"].mean(axis=1)
harsh_std = harsh_bundle["X"].std(axis=1)
enhanced_std = enhanced_bundle["X"].std(axis=1)

stats_df = pd.DataFrame({
    "harsh_mean": harsh_mean,
    "enhanced_mean": enhanced_mean,
    "harsh_std": harsh_std,
    "enhanced_std": enhanced_std,
})
stats_df.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(harsh_mean, bins=30, alpha=0.75, label="harsh", color="#4c78a8")
axes[0].hist(enhanced_mean, bins=30, alpha=0.60, label="enhanced", color="#f58518")
axes[0].set_title("Distribution of Mean Pixel Values")
axes[0].set_xlabel("Mean pixel value")
axes[0].legend()

axes[1].hist(harsh_std, bins=30, alpha=0.75, label="harsh", color="#4c78a8")
axes[1].hist(enhanced_std, bins=30, alpha=0.60, label="enhanced", color="#f58518")
axes[1].set_title("Distribution of Pixel Standard Deviation")
axes[1].set_xlabel("Pixel std")
axes[1].legend()
fig.tight_layout()


## Sample Images


#### Chạy thử nhận diên khuôn mặt

In [ ]:
plot_sample_images(
    harsh_bundle["X"],
    labels=harsh_bundle["y"],
    image_shape=image_shape,
    n_samples=config["n_samples"],
)


In [ ]:
plot_sample_images(
    enhanced_bundle["X"],
    labels=enhanced_bundle["y"],
    image_shape=image_shape,
    n_samples=config["n_samples"],
)


In [ ]:
plot_sample_images(
    harsh_bundle["X_test"],
    labels=harsh_bundle["y_test"],
    image_shape=image_shape,
    n_samples=config["n_samples"],
)


In [ ]:
plot_sample_images(
    enhanced_bundle["X_test"],
    labels=enhanced_bundle["y_test"],
    image_shape=image_shape,
    n_samples=config["n_samples"],
)


## Test Manifests


In [ ]:
pd.DataFrame({
    "harsh_test_file_path": harsh_bundle["test_file_paths"][:10],
    "enhanced_test_file_path": enhanced_bundle["test_file_paths"][:10],
})
